In [ ]:
import pandas as pd 
from langchain_community.llms import Ollama
from langchain_classic.agents import AgentExecutor, create_structured_chat_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.tools import tool

C:\Users\Moon\AppData\Local\Temp\ipykernel_16436\2166289764.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama


ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\miniconda\Lib\site-packages\langchain\agents\__init__.py)

Penser à pip install langchain

In [ ]:
#ici il nous faut une fonction qui transforme notre excel en data frame pandas
@tool
def lire_planning_excel(chemin_fichier: str):
    """ Utile pour lire le contenu du fichier Excel """
    return pd.read_excel(chemin_fichier)

In [ ]:
# Configuration d'Ollama et de l'Agent IA

llm = Ollama(model="llama3", temperature=0)

# Liste des outils mis à disposition de l'agent
tools = [lire_planning_excel]

# Création du prompt pour guider l'agent
system_prompt = """ Tu es un expert en ressources humaines et en planification de personnel.
Ton rôle est d'analyser les fichiers de planning, de comprendre les contraintes et d'aider à affecter les agents de manière optimale.
Tu as accès aux outils suivants : {tool_names}.
"""

#on fait un joli prompt avec notre prompt de base auquel on ajoute de la mémoire pour qu'il garde l'historique de la conversation, inupt est notre question (humaine) et le blocnotes c'est là où l'agent écrit ses pensées
prompt = ChatPromptTemplate.from_messages([("system", system_prompt), MessagesPlaceholder(variable_name="chat_history", optional=True),("human", "{input}\n\n{agent_scratchpad}"),])

#on construit l'agent en lui donnant tout
agent = create_structured_chat_agent(llm, tools, prompt)

# on l"exécute ! 
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,  # pour voir le raisonnement
    handle_parsing_errors=True
)